# Lesson 7 — The certificate of infeasibility is the trade

Rather than scanning for arbitrage shapes someone imagined, the engine asks whether any probability measure fits the quoted prices. When none does, linear-programming duality hands back the portfolio that profits in every state.

**The rule.** `max t subject to payoff − cost ≥ t; t* > 0 ⟹ the dual is the portfolio`

**When it holds.** For any set of constraints written as linear rows, which is why new instruments extend the engine by adding rows rather than code paths.

**When it fails.** A solver answer is only as good as the executability constraints in it. An LP that ignores resting depth, price grids or per-shard collateral returns a portfolio nobody can fill.

| | |
|---|---|
| Lesson id | `duality` |
| Pane it appears on | `certificate` (panes carry more than one lesson) |
| Code it is about | `modules/coherence/kernel/dutchbook.py` |
| Tests that go red if it stops being true | `tests/test_coherence_dutchbook.py` |
| Pane shipped | yes |

Every cell below runs against the real kernel. Nothing here is a re-implementation:
a number this notebook prints is the number the engine would produce for the same
input. The recorded Kalshi payloads come from `tests/fixtures/coherence/`.

In [ ]:
import json
import sys
from decimal import Decimal
from pathlib import Path

# This notebook lives in notebooks/coherence_lab/ and imports the kernel two
# levels up. Found by walking upward rather than by counting parents, so the
# notebook runs from its own directory or from Part2_Infrastructure.
HERE = Path.cwd().resolve()
ROOT = next((path for path in (HERE, *HERE.parents) if (path / "modules" / "coherence" / "kernel").is_dir()), None)
if ROOT is None:
    raise SystemExit(f"no coherence kernel above {HERE}: open this notebook from inside Part2_Infrastructure")
sys.path.insert(0, str(ROOT))

FIXTURES = ROOT / "tests" / "fixtures" / "coherence"


def fixture(name: str) -> dict:
    """One recorded Kalshi response, envelope and all, exactly as it was sent.

    These are captures, not mocks. Where a number below looks odd it is because
    the exchange quoted it, and `tools/capture_kalshi_fixtures.py` re-records
    them.
    """
    return json.loads((FIXTURES / f"{name}.json").read_text(encoding="utf-8"))


print(f"kernel root       {ROOT}")
print(f"recorded fixtures {FIXTURES.is_dir()}")

## 1. The solver seam, and what happens without it

In [ ]:
from modules.coherence.kernel import closedform, dutchbook
from modules.coherence.kernel.book import Book, Level
from modules.coherence.kernel.constraints import rows_for
from modules.coherence.kernel.costs import FeeSchedule
from modules.coherence.kernel.lattice import Component, Node

SCHEDULE = FeeSchedule()
optimize, absence_reason = dutchbook.import_linprog()

print(f"  linprog available : {dutchbook.linprog_available()}")
print(f"  reason if not     : {absence_reason}")
print()
print("  Nothing in this notebook imports scipy. `import_linprog()` is the only door, and")
print("  it is cached both ways: a missing package does not appear halfway through a")
print("  process, and retrying the import on every solve turns one absence into thousands")
print("  of failed imports. Where it is absent the engine falls back to the closed-form")
print("  checks and SAYS SO, because an absence must never look like present-and-fine.")

## 2. Three outcomes, and a dollar on sale at ninety cents

In [ ]:
def quote(ticker, yes_bid, no_bid, size=50_000):
    return Book(
        ticker=ticker,
        yes_bids=(Level(Decimal(yes_bid), size),),
        no_bids=(Level(Decimal(no_bid), size),),
    )

nodes = [
    Node(f"X-{index}", "X", "X", 0, "custom", None, None, ("S",), f"Outcome {index}")
    for index in (1, 2, 3)
]
family = Component(
    component_id="X", event_ticker="X", series_ticker="X",
    exchange_index=0, mutually_exclusive=True, nodes=nodes,
)
DUTCH = {f"X-{index}": quote(f"X-{index}", "0.2800", "0.7000") for index in (1, 2, 3)}

basket = sum((DUTCH[node.ticker].best_yes_ask for node in nodes), Decimal(0))
print("  three mutually exclusive outcomes, each bid 0.70 on the NO side")
print(f"  so each is offered at 1 - 0.70 = {DUTCH['X-1'].best_yes_ask}")
print(f"  and the basket costs {basket} for the dollar exactly one of them pays")

## 3. The closed-form engine, which always runs

In [ ]:
closed = closedform.solve(family, rows_for(family, DUTCH), SCHEDULE)
print(closed.render_text())

## 4. The linear programme, which runs where SciPy is

In [ ]:
lp = dutchbook.solve(family, DUTCH, SCHEDULE)
if lp is None:
    print("No linear programme ran here. The seam reported:")
    print(f"  {absence_reason}")
    print()
    print("The closed-form certificate above is the answer, and it finds strictly LESS:")
    print("it can only see a violation that fits inside one constraint row, never a")
    print("portfolio assembled across several. That is why the certificate names its")
    print("engine — so a reader can tell 'no arbitrage here' from 'none the weaker")
    print("engine can see'.")
else:
    print(lp.render_text())

## 5. Two engines on one question

In [ ]:
print(f"  closed form  {closed.verdict:<11} engine {closed.engine:<11} net {closed.net_edge}")
if lp is None:
    print("  highs        not run")
else:
    print(f"  highs        {lp.verdict:<11} engine {lp.engine:<11} net {lp.net_edge}")
    print()
    print(f"  the two agree to within {abs(lp.net_edge - closed.net_edge)}, which is the fill-count")
    print("  assumption and nothing else: the LP folds a per-contract trade fee into the")
    print("  prices it optimises over, then the cost model re-prices the winner exactly.")

## 6. A new instrument: Fréchet bounds on a parlay

In [ ]:
from modules.coherence.kernel.frechet import Combo, ComboLeg, assess, rows_for_combo

combo_books = {
    "A": quote("A", "0.4900", "0.4900"),
    "B": quote("B", "0.3900", "0.5900"),
    "PARLAY": quote("PARLAY", "0.5500", "0.4300"),
}
parlay = Combo(
    ticker="PARLAY", collection_ticker="C", exchange_index=0, label="both legs land",
    legs=(ComboLeg("A", "EA", "yes", "Leg A", 0), ComboLeg("B", "EB", "yes", "Leg B", 0)),
)
reading = assess(parlay, combo_books)

print(f"  legs quoted at        {[str(leg.probability) for leg in reading.legs]}")
print(f"  Fréchet band          [{reading.lower_bound}, {reading.upper_bound}], width {reading.band_width}")
print(f"  independence would say {reading.independence}")
print(f"  the parlay is quoted at {reading.combo_mid}; inside the band: {reading.inside_band}")
print(f"  {reading.detail}")
print()
print("  The band width is how far this price can move with no leg price moving at all.")
print("  Only a price OUTSIDE the band is a mispricing; where it sits inside is dependence,")
print("  and dependence is not quoted anywhere.")

## 7. New rows, no new code path

In [ ]:
combo_rows = rows_for_combo(parlay, combo_books)
shell = Component(
    component_id="PARLAY", event_ticker="PARLAY", series_ticker="C",
    exchange_index=0, mutually_exclusive=False, nodes=[],
)
print(f"  the combo produced {len(combo_rows)} rows in the same shape every other family uses:")
for row in combo_rows:
    print(f"    {row.family:<9} bound {row.bound}  cost {row.cost}  slack {row.slack}  violated {row.violated}")
print()
print(closedform.solve(shell, combo_rows, SCHEDULE).render_text())
print()
print("  No new code path was added to the solver. A new instrument extends this engine by")
print("  adding rows, which is the practical content of 'the dual is the trade'.")